In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

In [ ]:
df = pd.read_csv(
    'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
)

In [ ]:
print(df.head())

In [ ]:
print(df.info())

In [ ]:
print(df.describe())

In [ ]:
print(df.isnull().sum())

In [ ]:
df.drop('customerID', axis=1, inplace=True)

In [ ]:
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

In [ ]:
df['TotalCharges'].fillna(
    df['TotalCharges'].median(),
    inplace=True
)

In [ ]:
df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

In [ ]:
encoder = LabelEncoder()

categorical_cols = df.select_dtypes(
    include='object'
).columns

for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col])

In [ ]:
plt.figure(figsize=(14,10))

sns.heatmap(
    df.corr(),
    cmap='coolwarm'
)

plt.title('Correlation Heatmap')
plt.show()

In [ ]:
sns.countplot(x='Churn', data=df)

plt.title('Churn Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

sns.boxplot(
    x='Churn',
    y='MonthlyCharges',
    data=df
)

plt.title('Monthly Charges vs Churn')
plt.show()

In [ ]:
X = df.drop('Churn', axis=1)

y = df['Churn']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr_model = LogisticRegression()

lr_model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
y_pred_lr = lr_model.predict(
    X_test_scaled
)

In [ ]:
print(
    accuracy_score(y_test, y_pred_lr)
)

In [ ]:
print(
    confusion_matrix(y_test, y_pred_lr)
)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_lr
    )
)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
print(
    accuracy_score(y_test, y_pred_rf)
)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_rf
    )
)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

xgb_model.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

In [ ]:
print(
    accuracy_score(
        y_test,
        y_pred_xgb
    )
)

In [ ]:
print(
    roc_auc_score(
        y_test,
        y_pred_xgb
    )
)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_xgb
    )
)

In [ ]:
importance = xgb_model.feature_importances_

feature_names = X.columns

feature_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
})

feature_df = feature_df.sort_values(
    by='Importance',
    ascending=False
)

print(feature_df)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_df
)

plt.title('Feature Importance')
plt.show()

In [ ]:
print(
    'Logistic Regression Accuracy:',
    accuracy_score(y_test, y_pred_lr)
)

print(
    'Random Forest Accuracy:',
    accuracy_score(y_test, y_pred_rf)
)

print(
    'XGBoost Accuracy:',
    accuracy_score(y_test, y_pred_xgb)
)